# EZhire Gradio Setup
Loads saved models and metrics artifacts to run the dashboard.

In [ ]:
!pip install -q sentence-transformers scikit-learn plotly gradio nltk PyMuPDF einops

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
import nltk
import fitz
import torch
from collections import Counter
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = os.path.abspath(".")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
ARTIFACT_DIR = os.path.join(BASE_DIR, "artifacts")

STOP_WORDS = set(stopwords.words("english"))
print(f"Device: {DEVICE}")

## Load ensemble config and best SBERT

In [ ]:
CONFIG_PATH = os.path.join(ARTIFACT_DIR, 'ensemble_config.json')
best_sbert_name = 'mpnet'
BEST_W = 0.7

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
        cfg = json.load(f)
    best_sbert_name = cfg.get('best_sbert_name', best_sbert_name)
    BEST_W = float(cfg.get('best_sbert_weight', BEST_W))
else:
    print('Warning: ensemble_config.json not found, using defaults.')

SBERT_PATHS = {
    'mpnet': os.path.join(MODEL_DIR, 'ezhire-mpnet'),
    'roberta': os.path.join(MODEL_DIR, 'ezhire-roberta')
}

if best_sbert_name not in SBERT_PATHS:
    raise ValueError(f'Unknown SBERT name: {best_sbert_name}')

best_sbert_path = SBERT_PATHS[best_sbert_name]
if not os.path.exists(os.path.join(best_sbert_path, 'modules.json')):
    raise FileNotFoundError(f'Missing SBERT model at {best_sbert_path}. Run training notebooks first.')

best_sbert_model = SentenceTransformer(best_sbert_path, device=DEVICE)
best_sbert_model.max_seq_length = 384
print(f'Loaded best SBERT: {best_sbert_name} | weight={BEST_W:.2f}')

## Scoring helpers

In [ ]:
CHUNK_OVERLAP = 38
MAX_RESUME_CHUNKS = 10
MAX_JD_CHUNKS = 8
TOP_K_CHUNK_PAIRS = 5
ENCODE_BATCH = 16 if DEVICE == 'cuda' else 8

def raw_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t\f\v]+', ' ', l).strip() for l in text.split('\n')]
    return ' '.join(l for l in lines if l)

def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    tokens = word_tokenize(re.sub(r'\s+', ' ', text).strip())
    return ' '.join(t for t in tokens if t not in STOP_WORDS and len(t) > 1)

def extract_first_sentence(text, max_chars=150):
    t = raw_text(text)
    m = re.search(r'(?<=[a-zA-Z0-9])[.!?]', t)
    if m and m.start() > 10:
        sentence = t[:m.start() + 1].strip()
    else:
        sentence = t[:max_chars].strip()
    return sentence[:max_chars]

def token_chunk_body(body, tokenizer, max_tokens=512, overlap=CHUNK_OVERLAP, prefix_tokens=0):
    body = raw_text(body)
    if not body:
        return []
    ids = tokenizer.encode(body, add_special_tokens=False, truncation=False)
    effective_max = max(64, max_tokens - prefix_tokens - 2)
    if len(ids) <= effective_max:
        return [body]
    overlap = min(overlap, effective_max // 2)
    step = effective_max - overlap
    chunks = []
    for start in range(0, len(ids), step):
        piece = ids[start:start + effective_max]
        chunk = raw_text(tokenizer.decode(piece, skip_special_tokens=True))
        if chunk:
            chunks.append(chunk)
        if start + effective_max >= len(ids):
            break
    return chunks

def make_text_chunks(text, tokenizer, model_max_tokens=512, overlap=CHUNK_OVERLAP,
                     max_chunks=MAX_RESUME_CHUNKS, source_label='DOCUMENT'):
    text = raw_text(text)
    if not text:
        return []
    doc_ctx = extract_first_sentence(text)
    prefix = f'[DOC]: {doc_ctx} [{source_label}] ' if doc_ctx else f'[{source_label}] '
    prefix_tokens = len(tokenizer.encode(prefix, add_special_tokens=False))
    body_chunks = token_chunk_body(
        text, tokenizer,
        max_tokens=model_max_tokens,
        overlap=overlap,
        prefix_tokens=prefix_tokens
    )
    chunks = [f'{prefix}{c}' for c in body_chunks]
    if len(chunks) > max_chunks:
        keep = np.linspace(0, len(chunks) - 1, max_chunks, dtype=int).tolist()
        chunks = [chunks[i] for i in keep]
    fallback = text[:2000]
    return chunks or [f'{prefix}{fallback}']

def aggregate_chunk_sims(sims, top_k=TOP_K_CHUNK_PAIRS):
    sims = np.asarray(sims, float)
    if sims.size == 0: return 0.0
    flat = sims.reshape(-1)
    k = min(top_k, len(flat))
    top_mean = float(np.partition(flat, -k)[-k:].mean())
    coverage = float((sims.max(axis=0).mean() + sims.max(axis=1).mean()) / 2)
    return float(np.clip(0.75 * top_mean + 0.25 * coverage, 0.0, 1.0))

def score_doc_pair(model, t1, t2, left='RESUME', right='JOB'):
    tok = model.tokenizer
    mlen = model.max_seq_length
    c1 = make_text_chunks(t1, tok, model_max_tokens=mlen,
                          max_chunks=MAX_RESUME_CHUNKS, source_label=left)
    c2 = make_text_chunks(t2, tok, model_max_tokens=mlen,
                          max_chunks=MAX_JD_CHUNKS, source_label=right)
    e1 = model.encode(c1, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    e2 = model.encode(c2, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    sims = util.cos_sim(e1, e2).detach().cpu().numpy()
    return aggregate_chunk_sims(sims)

def tfidf_score(t1, t2):
    try:
        m = TfidfVectorizer().fit_transform([t1, t2])
        return float(cosine_similarity(m[0:1], m[1:2])[0][0])
    except Exception:
        return 0.0

def ensemble_score(s, t, sw=BEST_W):
    return round((sw * s + (1 - sw) * t) * 100, 2)

def get_tier(score):
    if score >= 70: return 'Strong Match'
    if score >= 45: return 'Potential Fit'
    return 'Poor Match'

def extract_pdf_text(path):
    try:
        return ' '.join(p.get_text() for p in fitz.open(path)).strip()
    except Exception as e:
        return f'PDF error: {e}'

def extract_keywords(text, n=30):
    try:
        vec = TfidfVectorizer(stop_words='english', max_features=n)
        vec.fit([text])
        return set(vec.get_feature_names_out())
    except Exception:
        tokens = word_tokenize(text.lower())
        return set(w for w, _ in Counter(
            t for t in tokens if t not in STOP_WORDS and len(t) > 2
        ).most_common(n))

def sbert_score_best(resume, jd):
    return score_doc_pair(best_sbert_model, resume, jd)

def score_single(resume, jd, sw=BEST_W):
    s = sbert_score_best(resume, jd)
    t = tfidf_score(clean_text(resume), clean_text(jd))
    return round(s * 100, 2), round(t * 100, 2), ensemble_score(s, t, sw)

## Load metrics artifacts (optional)

In [ ]:
metrics_path = os.path.join(ARTIFACT_DIR, 'metrics_df.csv')
scores_path = os.path.join(ARTIFACT_DIR, 'validation_scores.csv')

if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()
    print('Warning: metrics_df.csv not found, comparison plots will be empty.')

if os.path.exists(scores_path):
    df_sample = pd.read_csv(scores_path)
else:
    df_sample = pd.DataFrame()
    print('Warning: validation_scores.csv not found, heatmaps will be empty.')

In [ ]:
def tab1_score(resume_file, paste, jd, sw):
    resume = extract_pdf_text(resume_file.name) if resume_file else paste.strip()
    if not resume: return 'Please upload PDF or paste resume.', None, ''
    if not jd.strip(): return 'Please enter a job description.', None, ''
    s, t, e = score_single(resume, jd, sw)
    tier = get_tier(e)
    fig = go.Figure()
    for val, nm, col in zip([s, t, e],
                           [f'SBERT ({best_sbert_name})', 'TF-IDF', 'Ensemble'],
                           ['#4A90D9', '#E67E22', '#27AE60']):
        fig.add_trace(go.Bar(x=[nm], y=[val], marker_color=col,
                             text=[f'{val:.1f}%'], textposition='outside', name=nm))
    fig.update_layout(title='Score Breakdown', yaxis=dict(range=[0, 115]),
                      height=350, showlegend=False,
                      plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    kr = extract_keywords(resume, 40)
    kj = extract_keywords(jd, 40)
    matched, missing, extra = kr & kj, kj - kr, kr - kj
    html = (
        "<div style='font-family:sans-serif;padding:12px'>"
        "<h3 style='color:#27AE60'>Matched (" + str(len(matched)) + ")</h3>"
        "<p style='color:#27AE60'>" + (', '.join(sorted(matched)) or 'None') + "</p>"
        "<hr><h3 style='color:#E74C3C'>Missing from Resume (" + str(len(missing)) + ")</h3>"
        "<p style='color:#E74C3C'>" + (', '.join(sorted(missing)) or 'None') + "</p>"
        "<hr><h3 style='color:#F39C12'>Resume-Only (" + str(len(extra)) + ")</h3>"
        "<p style='color:#F39C12'>" + (', '.join(sorted(extra)) or 'None') + "</p></div>"
    )
    summary = (
        f'## {tier}\n\n'
        f'| Model | Score |\n|---|---|\n'
        f'| SBERT ({best_sbert_name}) | {s:.1f}% |\n'
        f'| TF-IDF | {t:.1f}% |\n'
        f'| **Ensemble** | **{e:.1f}%** |\n'
        f'| Keyword Overlap | {len(matched)}/{len(kj)} JD keywords |',
    )
    return summary, fig, html

def tab2_rank(files, jd, sw):
    if not files or not jd.strip(): return None, None
    rows = []
    for rf in files:
        text = extract_pdf_text(rf.name)
        name = os.path.basename(rf.name).replace('.pdf', '')
        s, t, e = score_single(text, jd, sw)
        rows.append(dict(Candidate=name, SBERT=s, TF_IDF=t, Ensemble=e, Tier=get_tier(e)))
    rdf = pd.DataFrame(rows).sort_values('Ensemble', ascending=False)
    rdf.insert(0, 'Rank', range(1, len(rdf) + 1))
    colors = ['#27AE60' if r >= 70 else '#F39C12' if r >= 45 else '#E74C3C'
              for r in rdf['Ensemble']]
    fig = go.Figure(go.Bar(x=rdf['Candidate'], y=rdf['Ensemble'],
                           marker_color=colors,
                           text=[f'{v:.1f}%' for v in rdf['Ensemble']],
                           textposition='outside'))
    fig.update_layout(title='Candidate Ranking', yaxis=dict(range=[0, 115]),
                      height=400, plot_bgcolor='rgba(0,0,0,0)',
                      paper_bgcolor='rgba(0,0,0,0)')
    return rdf[['Rank', 'Candidate', 'SBERT', 'TF_IDF', 'Ensemble', 'Tier']], fig

def tab3_heatmap(n=20):
    if df_sample.empty:
        blank = go.Figure().update_layout(title='No validation scores loaded.')
        return blank, blank
    n = min(int(n), len(df_sample))
    sub = df_sample.head(n).copy()
    sub['Label'] = [f'C{i + 1}' for i in range(n)]
    cols = ['mpnet_score', 'roberta_score', 'jina_score', 'tfidf_score', 'ensemble_score']
    z = sub[cols].values.T
    fh = go.Figure(go.Heatmap(z=z, x=sub['Label'].tolist(),
                                y=['mpnet', 'roberta', 'jina', 'TF-IDF', 'Ensemble'],
                                colorscale='RdYlGn', zmin=0, zmax=100,
                                text=np.round(z, 1), texttemplate='%{text}',
                                colorbar=dict(title='%')))
    fh.update_layout(title=f'Score Heatmap - {n} Candidates', height=380,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    fd = go.Figure()
    for col, nm, c in zip(cols,
                         ['mpnet', 'roberta', 'jina', 'TF-IDF', 'Ensemble'],
                         ['#4A90D9', '#9B59B6', '#E67E22', '#95A5A6', '#27AE60']):
        fd.add_trace(go.Histogram(x=df_sample[col], name=nm, opacity=0.6,
                                   marker_color=c, nbinsx=20))
    fd.update_layout(barmode='overlay', title='Score Distribution', height=340,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    return fh, fd

def tab4_compare():
    if metrics_df.empty:
        blank = go.Figure().update_layout(title='No metrics loaded.')
        return blank, blank, blank
    df = metrics_df.copy()
    if 'model' in df.columns:
        df = df.set_index('model')
    models = df.index.tolist()
    palette = ['#4A90D9', '#9B59B6', '#E67E22', '#95A5A6', '#27AE60']
    mcols = ['spearman', 'pearson', 'ndcg', 'precision_at_5', 'precision_at_10', 'mrr']
    fb = go.Figure()
    for i, m in enumerate(models):
        vals = [df.loc[m, c] for c in mcols]
        fb.add_trace(go.Bar(name=m,
            x=[c.replace('_', ' ').title() for c in mcols],
            y=vals, marker_color=palette[i % len(palette)],
            text=[f'{v:.3f}' for v in vals], textposition='outside'))
    fb.update_layout(barmode='group', title='All Metrics Comparison',
                     yaxis=dict(range=[-0.2, 1.3]), height=430,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    rcols = ['spearman', 'ndcg', 'precision_at_5', 'precision_at_10', 'mrr', 'pearson']
    fr = go.Figure()
    for i, m in enumerate(models):
        vals = [df.loc[m, c] for c in rcols] + [df.loc[m, rcols[0]]]
        cats = [c.replace('_', ' ').title() for c in rcols + [rcols[0]]]
        fr.add_trace(go.Scatterpolar(r=vals, theta=cats, fill='toself',
                                      name=m, line_color=palette[i % len(palette)], opacity=0.5))
    fr.update_layout(polar=dict(radialaxis=dict(range=[0, 1])),
                     title='Radar Chart', height=430,
                     paper_bgcolor='rgba(0,0,0,0)')
    fm = go.Figure(go.Bar(x=models,
        y=[df.loc[m, 'mae'] for m in models],
        marker_color=palette[:len(models)],
        text=[f"{df.loc[m, 'mae']:.4f}" for m in models],
        textposition='outside'))
    fm.update_layout(title='MAE norm (lower=better)', height=350,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    return fb, fr, fm

print('Dashboard functions ready')

In [ ]:
fig_bar_cmp, fig_radar_cmp, fig_mae_cmp = tab4_compare()
fig_heatmap, fig_dist = tab3_heatmap(20)

with gr.Blocks(theme=gr.themes.Soft(), title='EZhire') as demo:
    gr.Markdown('# EZhire - Resume-Job Semantic Similarity Scoring')
    gr.Markdown(
        f'Three-model pipeline: mpnet | roberta | jina  '
        f'| Ensemble uses **{best_sbert_name}** + TF-IDF '
        f'| Optimal SBERT weight: **{BEST_W:.2f}**'
    )

    sbert_w = gr.Slider(0.0, 1.0, value=float(BEST_W), step=0.05,
                        label='SBERT Weight (CV-optimised default shown)')

    with gr.Tab('Score a Resume'):
        with gr.Row():
            with gr.Column():
                r_pdf = gr.File(label='Upload PDF Resume', file_types=['.pdf'])
                r_paste = gr.Textbox(label='Or paste resume text', lines=7)
                jd_box = gr.Textbox(label='Job Description', lines=7)
                btn1 = gr.Button('Analyse Match', variant='primary')
            with gr.Column():
                out_md = gr.Markdown()
                out_fig = gr.Plot(label='Score Breakdown')
        out_kw = gr.HTML(label='Keyword Overlap')
        btn1.click(tab1_score, [r_pdf, r_paste, jd_box, sbert_w],
                   [out_md, out_fig, out_kw])

    with gr.Tab('Rank Candidates'):
        with gr.Row():
            with gr.Column():
                r_multi = gr.File(label='Upload Multiple PDFs',
                                   file_count='multiple', file_types=['.pdf'])
                jd_rank = gr.Textbox(label='Job Description', lines=7)
                btn2 = gr.Button('Rank Candidates', variant='primary')
            with gr.Column():
                rank_fig = gr.Plot()
        rank_tbl = gr.Dataframe(label='Ranked Candidates', interactive=False)
        btn2.click(tab2_rank, [r_multi, jd_rank, sbert_w], [rank_tbl, rank_fig])

    with gr.Tab('Score Heatmap'):
        n_sl = gr.Slider(5, min(50, len(df_sample)) if not df_sample.empty else 50,
                         value=20, step=5, label='Candidates to show')
        btn3 = gr.Button('Refresh')
        h_plot = gr.Plot(value=fig_heatmap)
        d_plot = gr.Plot(value=fig_dist)
        btn3.click(tab3_heatmap, [n_sl], [h_plot, d_plot])

    with gr.Tab('Model Comparison'):
        gr.Markdown(
            'All three models evaluated on the full validation set. '
            'Ensemble uses the best SBERT (auto-selected by Pearson) '
            'with CV-optimised TF-IDF weight.'
        )
        if not metrics_df.empty:
            gr.Dataframe(value=metrics_df, label='Metrics Table')
        gr.Plot(value=fig_bar_cmp, label='All Metrics')
        gr.Plot(value=fig_radar_cmp, label='Radar Chart')
        gr.Plot(value=fig_mae_cmp, label='MAE')
        gr.Markdown(
            '**Spearman/Pearson**: correlation with ATS ground truth. '
            '**NDCG**: ranking quality. **Precision@K**: good fits in top-K. '
            '**MAE**: prediction error. **RMSE/R2**: ATS-scale accuracy.'
        )

demo.launch(share=True, debug=True)